<a href="https://colab.research.google.com/github/yenuglaganesh-dev/Davp-project-/blob/main/Textbooks_in_Library_Analysis_Using_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", None)

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Libraries imported successfully.")

: 

In [ ]:
DATA_PATH = "library_textbooks.csv"  # Primary dataset file: "library_textbooks.csv" or "library_textbooks.xlsx"

if DATA_PATH and os.path.exists(DATA_PATH):
    if DATA_PATH.lower().endswith((".xlsx", ".xls")):
        raw = pd.read_excel(DATA_PATH)
    elif DATA_PATH.lower().endswith(".json"):
        raw = pd.read_json(DATA_PATH)
    else:
        raw = pd.read_csv(DATA_PATH)
    DATA_SOURCE = "user-provided dataset"
else:
    rng = np.random.default_rng(24)
    n = 700
    subjects = ["Mathematics", "Physics", "Chemistry", "Computer Science", "Electronics",
                "English", "Mechanical Engineering", "Biology", "Statistics", "Economics"]
    publishers = ["Pearson", "McGraw Hill", "Oxford University Press", "Wiley",
                  "Cambridge University Press", "S Chand", "Springer", "Tata McGraw-Hill"]
    titles = [
        "Engineering Mathematics", "Applied Physics", "Basic Chemistry", "Python Programming",
        "Digital Electronics", "Technical English", "Engineering Mechanics", "Data Structures",
        "Probability and Statistics", "Electrical Circuits", "Database Systems", "Thermodynamics",
        "Signals and Systems", "Computer Networks", "Linear Algebra", "Operating Systems",
        "Control Systems", "Microprocessors", "Discrete Mathematics", "Environmental Science"
    ]
    rows = []
    for i in range(n):
        subject = rng.choice(subjects)
        title = rng.choice(titles)
        publisher = rng.choice(publishers)
        year = int(rng.integers(2005, 2026))
        price = round(float(rng.choice([250, 300, 350, 400, 450, 500, 600, 750, 900]) * rng.uniform(.85, 1.15)), 2)
        copies = int(rng.integers(1, 26))
        available = int(rng.integers(0, copies + 1))
        rows.append({
            "BookID": f"BK{i+1:04d}",
            "Title": title,
            "Author": f"Author {int(rng.integers(1, 95)):02d}",
            "Subject": subject,
            "Publisher": publisher,
            "PublicationYear": year,
            "ISBN": f"978-{int(rng.integers(100000000, 999999999))}",
            "Price": price,
            "TotalCopies": copies,
            "AvailableCopies": available,
            "BorrowedCopies": copies - available,
            "Shelf": f"{chr(65 + int(rng.integers(0, 8)))}-{int(rng.integers(1, 31)):02d}"
        })
    raw = pd.DataFrame(rows)
    # Add small, deliberate data-quality issues for teaching missing-value handling.
    missing_idx = rng.choice(raw.index, size=14, replace=False)
    raw.loc[missing_idx[:4], "Publisher"] = np.nan
    raw.loc[missing_idx[4:8], "PublicationYear"] = np.nan
    raw.loc[missing_idx[8:11], "Price"] = np.nan
    raw.loc[missing_idx[11:], "AvailableCopies"] = np.nan
    DATA_SOURCE = "generated demo inventory (illustrative only)"

print("Data source:", DATA_SOURCE)
print("Rows:", raw.shape[0], "| Columns:", raw.shape[1])
display(raw.head())

In [ ]:
data = raw.copy()

rename_map = {}
for col in data.columns:
    key = " ".join(str(col).strip().lower().replace("_", " ").replace("-", " ").split())
    if key in ["book id", "bookid", "record id", "id"]:
        rename_map[col] = "BookID"
    elif key in ["title", "book title", "name"]:
        rename_map[col] = "Title"
    elif key in ["author", "authors", "writer"]:
        rename_map[col] = "Author"
    elif key in ["subject", "department", "category", "course"]:
        rename_map[col] = "Subject"
    elif key in ["publisher", "publishing company"]:
        rename_map[col] = "Publisher"
    elif key in ["publication year", "year", "published year", "year published"]:
        rename_map[col] = "PublicationYear"
    elif key in ["isbn", "isbn number"]:
        rename_map[col] = "ISBN"
    elif key in ["price", "book price", "cost", "unit price"]:
        rename_map[col] = "Price"
    elif key in ["total copies", "copies", "quantity", "stock"]:
        rename_map[col] = "TotalCopies"
    elif key in ["available copies", "available", "available stock"]:
        rename_map[col] = "AvailableCopies"
    elif key in ["borrowed copies", "borrowed", "issued copies", "issued"]:
        rename_map[col] = "BorrowedCopies"
    elif key in ["shelf", "shelf location", "location", "rack"]:
        rename_map[col] = "Shelf"

data = data.rename(columns=rename_map)
print("Standardized columns:", data.columns.tolist())
display(data.head())

In [ ]:
print("Rows:", data.shape[0])
print("Columns:", data.shape[1])
display(data.dtypes.to_frame("data_type"))
data.info()
display(data.head())
display(data.sample(min(5, len(data)), random_state=42))

In [ ]:
data = data.replace(["?", "NA", "N/A", "null", "None", ""], np.nan)

missing_summary = pd.DataFrame({
    "missing_count": data.isna().sum(),
    "missing_percent": (data.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)
display(missing_summary)
print("Total missing cells:", int(data.isna().sum().sum()))
print("Exact duplicate rows:", int(data.duplicated().sum()))

plt.figure(figsize=(12, 4))
sns.heatmap(data.isna(), cbar=False, yticklabels=False)
plt.title("Missing Values Map")
plt.xlabel("Columns")
plt.tight_layout()
plt.show()

In [ ]:
# Convert numeric columns carefully.
for col in ["PublicationYear", "Price", "TotalCopies", "AvailableCopies", "BorrowedCopies"]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

before = len(data)
data = data.drop_duplicates().copy()
print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", len(data))

# Fill missing text fields with a transparent placeholder.
for col in ["Title", "Author", "Subject", "Publisher", "Shelf", "BookID", "ISBN"]:
    if col in data.columns:
        data[col] = data[col].fillna("Unknown")

# Median-impute numeric fields for demonstration; in real inventory work, verify source records first.
for col in ["PublicationYear", "Price", "TotalCopies", "AvailableCopies", "BorrowedCopies"]:
    if col in data.columns and data[col].isna().any():
        data[col] = data[col].fillna(data[col].median())

print("Missing values after treatment:")
display(data.isna().sum().to_frame("missing_count"))

In [ ]:
if {"TotalCopies", "AvailableCopies"}.issubset(data.columns):
    data["AvailabilityRatePct"] = np.where(
        data["TotalCopies"] > 0,
        data["AvailableCopies"] / data["TotalCopies"] * 100,
        np.nan
    )
    data["BorrowedCopiesDerived"] = data["TotalCopies"] - data["AvailableCopies"]
    data["AvailabilityCheck"] = (
        (data["AvailableCopies"] < 0) |
        (data["TotalCopies"] < 0) |
        (data["AvailableCopies"] > data["TotalCopies"])
    )
    print("Records with invalid copy counts:", int(data["AvailabilityCheck"].sum()))

if {"Price", "TotalCopies"}.issubset(data.columns):
    data["EstimatedInventoryValue"] = data["Price"] * data["TotalCopies"]

if "PublicationYear" in data.columns:
    current_year = pd.Timestamp.today().year
    data["BookAgeYears"] = current_year - data["PublicationYear"]

display(data.head())

In [ ]:
numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
numeric_array = data[numeric_cols].to_numpy()
print("Numeric array shape:", numeric_array.shape)
print("First row:", numeric_array[0])
print("First five rows, first three numeric columns:\n", numeric_array[:5, :3])

if "Price" in data.columns:
    price_array = data["Price"].to_numpy()
    print("First five prices:", price_array[:5])
    print("First five prices with 10% increase:", (price_array[:5] * 1.10).round(2))
    print("Mean price with NumPy:", round(float(np.nanmean(price_array)), 2))

In [ ]:
if "Price" in data.columns:
    price_series = data["Price"]
    print("Series type:", type(price_series))
    display(price_series.head())

display(data.iloc[:5, :min(6, data.shape[1])])

if "Subject" in data.columns:
    subject_filter = data[data["Subject"].astype(str).str.contains("Computer", case=False, na=False)]
    print("Records in subjects containing 'Computer':", len(subject_filter))
    display(subject_filter.head())

In [ ]:
numeric_summary_cols = [c for c in ["PublicationYear", "Price", "TotalCopies", "AvailableCopies", "BorrowedCopies",
                                    "AvailabilityRatePct", "EstimatedInventoryValue", "BookAgeYears"] if c in data.columns]
display(data[numeric_summary_cols].describe().T)

kpis = {
    "Book records": len(data),
    "Unique titles": data["Title"].nunique() if "Title" in data.columns else np.nan,
    "Subjects": data["Subject"].nunique() if "Subject" in data.columns else np.nan,
    "Publishers": data["Publisher"].nunique() if "Publisher" in data.columns else np.nan,
    "Total copies": data["TotalCopies"].sum() if "TotalCopies" in data.columns else np.nan,
    "Available copies": data["AvailableCopies"].sum() if "AvailableCopies" in data.columns else np.nan,
    "Borrowed copies": data["BorrowedCopies"].sum() if "BorrowedCopies" in data.columns else np.nan,
    "Average price": round(float(data["Price"].mean()), 2) if "Price" in data.columns else np.nan,
    "Estimated inventory value": round(float(data["EstimatedInventoryValue"].sum()), 2) if "EstimatedInventoryValue" in data.columns else np.nan
}
display(pd.DataFrame({"Metric": list(kpis.keys()), "Value": list(kpis.values())}))

In [ ]:
if "Subject" in data.columns:
    subject_counts = data["Subject"].value_counts()
    display(subject_counts.to_frame("book_records"))

    plt.figure(figsize=(10, 5))
    subject_counts.sort_values().plot(kind="barh")
    plt.title("Number of Book Records by Subject")
    plt.xlabel("Number of records")
    plt.ylabel("Subject")
    plt.tight_layout()
    plt.show()

    if "AvailableCopies" in data.columns:
        subject_stock = data.groupby("Subject")[["TotalCopies", "AvailableCopies"]].sum().sort_values("TotalCopies", ascending=False)
        display(subject_stock)
        subject_stock.sort_values("TotalCopies").plot(kind="barh", figsize=(10, 6))
        plt.title("Total and Available Copies by Subject")
        plt.xlabel("Copies")
        plt.tight_layout()
        plt.show()

In [ ]:
if "Publisher" in data.columns:
    publisher_summary = data.groupby("Publisher").agg(
        book_records=("Title", "size") if "Title" in data.columns else ("Publisher", "size"),
        total_copies=("TotalCopies", "sum") if "TotalCopies" in data.columns else ("Publisher", "size"),
        average_price=("Price", "mean") if "Price" in data.columns else ("Publisher", "size")
    ).sort_values("book_records", ascending=False)
    display(publisher_summary)

    publisher_summary["book_records"].sort_values().plot(kind="barh", figsize=(9, 5))
    plt.title("Book Records by Publisher")
    plt.xlabel("Records")
    plt.tight_layout()
    plt.show()

In [ ]:
if "PublicationYear" in data.columns:
    year_counts = data["PublicationYear"].dropna().astype(int).value_counts().sort_index()
    plt.figure(figsize=(11, 4))
    plt.plot(year_counts.index, year_counts.values, marker="o")
    plt.title("Books by Publication Year")
    plt.xlabel("Publication year")
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.show()
    display(year_counts.to_frame("book_records").tail(15))

if "BookAgeYears" in data.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=data, x="BookAgeYears", bins=20, color="steelblue")
    plt.title("Distribution of Book Age")
    plt.xlabel("Book age (years)")
    plt.tight_layout()
    plt.show()

In [ ]:
if {"Subject", "TotalCopies", "AvailableCopies"}.issubset(data.columns):
    availability = data.groupby("Subject").agg(
        total_copies=("TotalCopies", "sum"),
        available_copies=("AvailableCopies", "sum")
    )
    availability["borrowed_or_unavailable_copies"] = availability["total_copies"] - availability["available_copies"]
    availability["availability_pct"] = np.where(
        availability["total_copies"] > 0,
        100 * availability["available_copies"] / availability["total_copies"],
        np.nan
    )
    display(availability.sort_values("availability_pct"))

    plot_avail = availability.sort_values("availability_pct")
    plt.figure(figsize=(10, 5))
    plt.barh(plot_avail.index, plot_avail["availability_pct"])
    plt.axvline(50, linestyle="--", label="50% reference")
    plt.title("Availability Rate by Subject")
    plt.xlabel("Available copies (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Copy availability fields are missing; availability analysis is skipped.")

In [ ]:
outlier_rows = []
for col in [c for c in ["Price", "TotalCopies", "AvailableCopies", "PublicationYear"] if c in data.columns]:
    series = data[col].dropna()
    if len(series) == 0:
        continue
    q1, q3 = series.quantile([0.25, 0.75])
    iqr_value = q3 - q1
    low, high = q1 - 1.5 * iqr_value, q3 + 1.5 * iqr_value
    mask = (data[col] < low) | (data[col] > high)
    outlier_rows.append({"column": col, "Q1": q1, "Q3": q3, "IQR": iqr_value,
                         "lower_bound": low, "upper_bound": high,
                         "potential_outliers": int(mask.sum())})
display(pd.DataFrame(outlier_rows).round(2))

for col in [c for c in ["Price", "TotalCopies", "PublicationYear"] if c in data.columns]:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=data[col], color="steelblue")
    plt.title(f"Potential Outliers: {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
corr_cols = [c for c in ["PublicationYear", "Price", "TotalCopies", "AvailableCopies",
                         "BorrowedCopies", "AvailabilityRatePct", "BookAgeYears"] if c in data.columns]
corr_matrix = data[corr_cols].corr()
cov_matrix = data[corr_cols].cov()
print("Correlation matrix:")
display(corr_matrix.round(3))
print("Covariance matrix:")
display(cov_matrix.round(3))

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Library Inventory Features")
plt.tight_layout()
plt.show()

In [ ]:
if "PublicationYear" in data.columns:
    year_counts = data["PublicationYear"].dropna().astype(int).value_counts().sort_index()
    plt.figure(figsize=(10, 4))
    plt.plot(year_counts.index, year_counts.values, marker="o", label="Book records")
    plt.title("Publication Year Trend")
    plt.xlabel("Year")
    plt.ylabel("Number of records")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

if "Subject" in data.columns and "TotalCopies" in data.columns:
    subject_copies = data.groupby("Subject")["TotalCopies"].sum().sort_values(ascending=False).head(10).sort_values()
    plt.figure(figsize=(9, 5))
    plt.barh(subject_copies.index, subject_copies.values)
    plt.title("Top Subjects by Total Copies")
    plt.xlabel("Total copies")
    plt.tight_layout()
    plt.show()

if "Price" in data.columns:
    plt.figure(figsize=(8, 4))
    plt.hist(data["Price"].dropna(), bins=20, edgecolor="black")
    plt.title("Textbook Price Distribution")
    plt.xlabel("Price")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

if {"Price", "TotalCopies"}.issubset(data.columns):
    plt.figure(figsize=(8, 5))
    plt.scatter(data["Price"], data["TotalCopies"], alpha=0.6)
    plt.title("Book Price vs Total Copies")
    plt.xlabel("Price")
    plt.ylabel("Total copies")
    plt.tight_layout()
    plt.show()

In [ ]:
if {"Subject", "Price"}.issubset(data.columns):
    common_subjects = data["Subject"].value_counts().head(8).index
    plt.figure(figsize=(11, 5))
    sns.boxplot(data=data[data["Subject"].isin(common_subjects)], x="Subject", y="Price", color="steelblue")
    plt.title("Price Distribution by Subject")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

if "Publisher" in data.columns:
    top_publishers = data["Publisher"].value_counts().head(8).index
    plt.figure(figsize=(10, 5))
    sns.countplot(data=data[data["Publisher"].isin(top_publishers)], y="Publisher",
                  order=top_publishers, color="steelblue")
    plt.title("Most Represented Publishers")
    plt.xlabel("Book records")
    plt.tight_layout()
    plt.show()

In [ ]:
if {"Price", "TotalCopies"}.issubset(data.columns):
    fig = px.scatter(
        data, x="Price", y="TotalCopies",
        color="Subject" if "Subject" in data.columns else None,
        hover_data=[c for c in ["Title", "Publisher", "PublicationYear", "AvailableCopies"] if c in data.columns],
        title="Interactive: Price vs Total Copies"
    )
    fig.show()

if "Subject" in data.columns and "TotalCopies" in data.columns:
    plot_subjects = data.groupby("Subject", as_index=False)["TotalCopies"].sum().sort_values("TotalCopies", ascending=False)
    fig = px.bar(plot_subjects, x="Subject", y="TotalCopies", title="Interactive Total Copies by Subject")
    fig.show()

if "PublicationYear" in data.columns:
    fig = px.histogram(data, x="PublicationYear", color="Subject" if "Subject" in data.columns else None,
                       title="Interactive Distribution of Publication Years")
    fig.show()

In [ ]:
dashboard_metric = "Price"
if dashboard_metric in data.columns:
    fig = px.histogram(
        data, x=dashboard_metric,
        color="Subject" if "Subject" in data.columns else None,
        marginal="box", title=f"Library Inventory Dashboard: {dashboard_metric}"
    )
    fig.show()
else:
    print("Choose a metric available in the dataset.")
print("Dashboard metric:", dashboard_metric)

In [ ]:
print("TEXTBOOKS IN LIBRARY ANALYSIS SUMMARY")
print("=" * 55)
print("Data source:", DATA_SOURCE)
print("Book records analyzed:", len(data))
print("Columns available:", len(data.columns))
print("Missing cells remaining:", int(data.isna().sum().sum()))
print("Duplicate rows remaining:", int(data.duplicated().sum()))

if "Title" in data.columns:
    print("Unique titles:", data["Title"].nunique())
if "TotalCopies" in data.columns:
    print("Total copies:", int(data["TotalCopies"].sum()))
if "AvailableCopies" in data.columns:
    print("Available copies:", int(data["AvailableCopies"].sum()))
if "BorrowedCopies" in data.columns:
    print("Borrowed copies:", int(data["BorrowedCopies"].sum()))
if "Price" in data.columns:
    print("Average textbook price:", round(float(data["Price"].mean()), 2))
if "EstimatedInventoryValue" in data.columns:
    print("Estimated inventory value:", round(float(data["EstimatedInventoryValue"].sum()), 2))

if "Subject" in data.columns:
    print("\nSubject with most records:")
    display(data["Subject"].value_counts().head(1).to_frame("book_records"))
if "Publisher" in data.columns:
    print("\nMost represented publisher:")
    display(data["Publisher"].value_counts().head(1).to_frame("book_records"))
if "AvailabilityRatePct" in data.columns:
    subject_avail = data.groupby("Subject")["AvailabilityRatePct"].mean().sort_values()
    print("\nSubject with lowest average record-level availability rate:")
    display(subject_avail.head(1).to_frame("average_availability_pct"))